In [1]:
import scanpy as sc
import pandas as pd
import numpy as np


In [4]:
adata = sc.read_h5ad('../data/mtDNA_DSB_5k_clustered_manual_annotation.h5ad')

/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [12]:
adata_raw = sc.read_h5ad('../data/mtDNA_DSB_5k_raw.h5ad')

/Users/christoffer/miniconda3/envs/sc/lib/python3.8/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [15]:
import numpy as np
import pandas as pd
from scipy import sparse

def _report_dups(name, idx):
    dup = pd.Index(idx)
    d = dup[dup.duplicated()].unique().tolist()
    if d:
        print(f"[WARN] {name}: {len(d)} duplicated names e.g. {d[:5]}")
    else:
        print(f"[OK] {name}: all unique")

def attach_counts_intersection_safe(adata, adata_raw, layer_name="counts", collapse_var=False):
    # 1) report duplicates
    _report_dups("adata.obs_names", adata.obs_names)
    _report_dups("adata.var_names", adata.var_names)
    _report_dups("adata_raw.obs_names", adata_raw.obs_names)
    _report_dups("adata_raw.var_names", adata_raw.var_names)

    # 2) handle duplicates
    if collapse_var:
        # collapse duplicated var names by summing columns (good for counts)
        def _collapse(A, names):
            df = pd.DataFrame.sparse.from_spmatrix(A) if sparse.issparse(A) else pd.DataFrame(A)
            df.columns = names
            return df.groupby(level=0, axis=1).sum()
        # collapse both objects on var (genes)
        A1 = _collapse(adata.X, adata.var_names)
        A2 = _collapse(adata_raw.X, adata_raw.var_names)
        # keep obs in original order
        A1.index = adata.obs_names
        A2.index = adata_raw.obs_names
        # replace X and var_names with collapsed
        adata = adata[:, []].copy()
        adata.X = A1.values
        adata.var_names = A1.columns.astype(str)
        adata.obs_names = A1.index.astype(str)

        adata_raw = adata_raw[:, []].copy()
        adata_raw.X = A2.values
        adata_raw.var_names = A2.columns.astype(str)
        adata_raw.obs_names = A2.index.astype(str)
    else:
        # just force uniqueness by appending suffixes
        adata.var_names_make_unique()
        adata.obs_names_make_unique()
        adata_raw.var_names_make_unique()
        adata_raw.obs_names_make_unique()

    # 3) intersect axes
    common_obs = adata.obs_names.intersection(adata_raw.obs_names)
    common_var = adata.var_names.intersection(adata_raw.var_names)

    ad_sub  = adata[common_obs, common_var].copy()
    raw_sub = adata_raw[common_obs, common_var]

    # 4) attach raw into a layer
    Xraw = raw_sub.X
    if sparse.issparse(Xraw):
        Xraw = Xraw.copy()
    else:
        Xraw = np.asarray(Xraw).copy()
    ad_sub.layers[layer_name] = Xraw

    print(f"[attach_counts] kept {ad_sub.n_obs} cells and {ad_sub.n_vars} genes "
          f"(intersected). Raw stored in .layers['{layer_name}'].")
    return ad_sub

In [16]:
# If you want to simply make names unique (fastest):
adata = attach_counts_intersection_safe(adata, adata_raw, layer_name="counts", collapse_var=False)

# If your var (gene) names are symbols with duplicates and you prefer to SUM duplicates first:
# adata = attach_counts_intersection_safe(adata, adata_raw, layer_name="counts", collapse_var=True)

[WARN] adata.obs_names: 96 duplicated names e.g. ['aggekfip-1', 'lapokbab-1', 'dilbcmig-1', 'aabdhpop-1', 'ajnddggj-1']
[OK] adata.var_names: all unique
[WARN] adata_raw.obs_names: 98 duplicated names e.g. ['aggekfip-1', 'lapokbab-1', 'dilbcmig-1', 'aabdhpop-1', 'ajnddggj-1']
[OK] adata_raw.var_names: all unique
[attach_counts] kept 980474 cells and 5101 genes (intersected). Raw stored in .layers['counts'].


In [20]:
adata.layers['counts']

<980474x5101 sparse matrix of type '<class 'numpy.float32'>'
	with 545822587 stored elements in Compressed Sparse Row format>

In [88]:
adata_OL = adata[adata.obs.cell_class.str.contains('ligo')]

In [89]:
adata_sub = adata_OL[adata_OL.obs["condition"]=='mtDSB']
adata_sub = adata_sub[adata_sub.obs["age"]=='60']

adata_rest = adata_OL[adata_OL.obs["condition"]!='mtDSB']
adata_rest = adata_rest[adata_rest.obs["age"]=='60']

In [95]:

# mean expression per gene for each group
mean_interest = np.asarray(adata_sub.layers['counts'].mean(axis=0)).ravel()
mean_rest = np.asarray(adata_rest.layers['counts'].mean(axis=0)).ravel()

diff = mean_interest - mean_rest
fold = (mean_interest + 1e-6) / (mean_rest + 1e-6)

diff_df = pd.DataFrame({
    "gene": adata.var_names,
    "mean_interest": mean_interest,
    "mean_rest": mean_rest,
    "diff": diff,
    "log2FC": np.log2(fold)
}).sort_values("log2FC", ascending=False)

In [103]:
diff_df_sorted = diff_df.sort_values(by=[ "mean_interest",'log2FC'], ascending=[False, False])
print(diff_df_sorted[diff_df_sorted.log2FC > 0.3].head(60))

          gene  mean_interest  mean_rest      diff    log2FC
2502     Kif5a       3.009242   2.323457  0.685785  0.373127
3056     Ndrg2       1.989233   1.575153  0.414080  0.336720
2953       Mt2       1.598181   0.970077  0.628104  0.720259
2503     Kif5b       1.559481   1.215348  0.344133  0.359696
1536   Fam107a       1.370666   1.020260  0.350407  0.425941
1988     Gstp1       1.196373   0.787507  0.408866  0.603301
2421       Jun       1.093714   0.878043  0.215671  0.316872
3148       Nmu       1.078806   0.163056  0.915750  2.725985
380        B2m       1.039785   0.765304  0.274480  0.442179
2182     Hsph1       1.037044   0.766774  0.270270  0.435602
1814      Gfap       0.954771   0.423558  0.531212  1.172591
2176     Hspa5       0.902121   0.675725  0.226396  0.416884
330       Atf4       0.817541   0.636568  0.180973  0.360976
3243     Ntsr2       0.768345   0.620689  0.147656  0.307882
3763     Ptpra       0.708830   0.565199  0.143631  0.326679
2180     Hspd1       0.6

In [104]:
diff_df_sorted = diff_df.sort_values(by=['log2FC',"mean_interest"], ascending=[False, False])
print(diff_df_sorted[diff_df_sorted.log2FC > 0.2].head(60))

           gene  mean_interest  mean_rest      diff    log2FC
4268    Slc5a12       0.000130   0.000000  0.000130  7.030532
4778      Trib3       0.036845   0.001479  0.035366  4.637878
690        Cd40       0.030933   0.001588  0.029345  4.283258
1804      Gdf15       0.009211   0.000500  0.008711  4.200003
2265    Il13ra2       0.000278   0.000022  0.000256  3.616405
1056       Cst7       0.009786   0.001022  0.008764  3.257730
2135      Hoxb5       0.001390   0.000152  0.001238  3.182244
764      Cdkn1a       0.056972   0.006329  0.050643  3.169980
3307        Oxt       0.001612   0.000217  0.001395  2.884480
4034      Scimp       0.000315   0.000043  0.000272  2.828431
3148        Nmu       1.078806   0.163056  0.915750  2.725985
663        Cd22       0.000853   0.000130  0.000722  2.698464
2133      Hoxb3       0.000964   0.000174  0.000790  2.462855
883      Cldn17       0.000241   0.000043  0.000197  2.442811
4118   Serpine1       0.003188   0.000587  0.002601  2.438565
3636    

## Functional Modules in mtDSB Oligodendrocytes

### 1. Oxidative Stress & Detoxification
- **Mt2** – metallothionein, binds Zn/Cu, scavenges ROS  
- **Gstp1** – glutathione detox enzyme  
- **Sqstm1 (p62)** – oxidative stress sensor, links ROS to autophagy  
- **Nfe2l1** – TF regulating antioxidant genes  
- **Hspd1, Hspa9** – mitochondrial chaperones for ROS stress  
➡️ Evidence for **oxidative stress and mitochondrial redox imbalance**

---

### 2. Integrated Stress Response (ISR) & UPRmt
- **Atf4, Jun** – stress-activated TFs driving ISR/UPRmt  
- **Hspa5 (BiP/GRP78), Hspd1 (HSP60), Hspa9** – ER/mitochondrial chaperones  
- **Hsph1** – HSP110 family, protein folding/stress tolerance  
➡️ Indicates **protein misfolding and translational stress downstream of mitochondrial dysfunction**

---

### 3. Antigen Presentation & Immune Signaling
- **B2m, H2-D1, H2-K1** – MHC-I antigen presentation  
- **Ctss (Cathepsin S)** – lysosomal protease for antigen processing  
- **Cd40, Cd44, Cd22, Cd3e** – co-stimulatory/immune interaction molecules  
➡️ Suggests **stressed OLs present antigen and engage immune surveillance**

---

### 4. Axonal/Myelin Transport Stress
- **Kif5a, Kif5b** – kinesin motors for axonal/myelin cargo  
- **Dync1li1** – dynein light intermediate chain, retrograde transport  
- **Ptpra, Itgb1, Mpzl1, Sorbs1** – adhesion/cell interaction molecules  
➡️ Points to **disturbed axonal transport and OL–axon coupling**

---

### 5. Astrocytic Reactivity & Glial Crosstalk
- **Ndrg2, Gfap, Mlc1** – hallmark astrocytic/reactive gliosis genes  
- **S100a1, Calb2** – calcium-binding proteins in reactive astrocytes  
➡️ Reflects **astrocyte activation secondary to OL mtDSB stress**

---

### 6. Inflammatory Signaling & Cytokines
- **Nmu (Neuromedin U)** – neuropeptide with pro-inflammatory activity  
- **Ccl3 (MIP-1α)** – chemokine recruiting myeloid cells  
- **Irf4** – immune TF controlling cytokine expression  
➡️ Suggests **immune-modulatory signaling in the OL/astro niche**

---

### 7. Metabolic Remodeling
- **Hadhb** – mitochondrial β-oxidation enzyme  
- **Parvb, Sorbs1** – cytoskeletal/metabolic adaptors  
- **Fosl1, Myc** – TFs linked to metabolic reprogramming and proliferation  
➡️ Evidence for **shifts in mitochondrial and metabolic regulation**

---

## ✅ Overall Takeaway
mtDSB oligodendrocytes show a **coherent multi-pathway stress program**:
- **Redox stress** → metallothioneins, glutathione enzymes  
- **Mitochondrial/ER stress** → ISR/UPRmt activation  
- **Immune presentation** → MHC-I and lysosomal proteases  
- **Axonal transport disruption** → kinesins/dyneins  
- **Astro reactivity** → Ndrg2, Gfap upregulation  
- **Pro-inflammatory signaling** → Nmu, Ccl3  
➡️ Together, these changes suggest OLs under mtDNA damage are **alive but stressed**, tipping the microenvironment toward **immune activation and glial crosstalk**.